In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from sympy import pprint

import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'scripts'))

import importlib
import opg2
importlib.reload(opg2)
from opg2 import s_vec

from l_print import lPrint

![image.png](./../image%20copy%203.png)

#### 1)
Først konstruer vi en global positions vektor for $C_2$

In [2]:

# Kendte størrelser
d, l2, m2, i2, m1 , g= sp.symbols('d l2 m2 i2 m1, g')


i2 = (m2*l2**2)/12

# Variable størrelser
t = sp.symbols('t')

theta2 = sp.Function('theta2')(t)
d1 = sp.Function('d1')(t)
v, omega = sp.symbols('v omega')

r_c2 = sp.Matrix([d1, -d, 0]) + sp.Matrix([l2/2*sp.sin(theta2), -l2/2*sp.cos(theta2), 0])
v_c2 = sp.diff(r_c2, t)

lPrint("Position af C2", r_c2)
lPrint("Hastighed af C2", v_c2)




**Position af C2**: $$\left[\begin{matrix}\frac{l_{2} \sin{\left(\theta_{2}{\left(t \right)} \right)}}{2} + d_{1}{\left(t \right)}\\- d - \frac{l_{2} \cos{\left(\theta_{2}{\left(t \right)} \right)}}{2}\\0\end{matrix}\right]$$

**Hastighed af C2**: $$\left[\begin{matrix}\frac{l_{2} \cos{\left(\theta_{2}{\left(t \right)} \right)} \frac{d}{d t} \theta_{2}{\left(t \right)}}{2} + \frac{d}{d t} d_{1}{\left(t \right)}\\\frac{l_{2} \sin{\left(\theta_{2}{\left(t \right)} \right)} \frac{d}{d t} \theta_{2}{\left(t \right)}}{2}\\0\end{matrix}\right]$$

#### 2)
Starter med kinetisk energi for translation:
$$
T_{trans} = \frac{1}{2} m v^2 
$$

In [3]:
T_1 = 1/2 * m1 * sp.diff(d1, t)**2
lPrint("Kinetisk energi for blok", T_1)


**Kinetisk energi for blok**: $$0.5 m_{1} \left(\frac{d}{d t} d_{1}{\left(t \right)}\right)^{2}$$

Den potiontielle energi for blokken er bare 0, da højden er 0.

In [4]:
V_1 = 0

#### 3)

Den kinetiske energi for link AB er givet ved general motion:
$$
T_{general} = \frac{1}{2} I_G \omega^2 + \frac{1}{2} m v_G^2
$$

In [5]:
T_2 = 1/2 * i2 * sp.diff(theta2, t)**2 + 1/2 * m2 * (v_c2.dot(v_c2))
lPrint("Kinetisk energi for link AB", T_2.simplify())

**Kinetisk energi for link AB**: $$m_{2} \left(0.166666666666667 l_{2}^{2} \left(\frac{d}{d t} \theta_{2}{\left(t \right)}\right)^{2} + 0.5 l_{2} \cos{\left(\theta_{2}{\left(t \right)} \right)} \frac{d}{d t} d_{1}{\left(t \right)} \frac{d}{d t} \theta_{2}{\left(t \right)} + 0.5 \left(\frac{d}{d t} d_{1}{\left(t \right)}\right)^{2}\right)$$

Vi definerer vores reference punkt for den potiontielle energi til at være y værdigen når $\theta = 0$ 

In [6]:

v, omega = sp.symbols('v omega')

i2 = (m2*l2**2)/3

# Substituerer fart og vinkelhastighed

V_2  = m2 * g * r_c2[1]
lPrint("Potiontiel energi for link AB", V_2.simplify())

**Potiontiel energi for link AB**: $$- \frac{g m_{2} \left(2 d + l_{2} \cos{\left(\theta_{2}{\left(t \right)} \right)}\right)}{2}$$

#### 4)
To finde the legrangian we just do this:
$$
L = T_1 - V_1 + T_2 - V_2
$$


In [7]:
L = T_1 - V_1 + T_2 - V_2

lPrint("Lagrangian L", L)

**Lagrangian L**: $$- g m_{2} \left(- d - \frac{l_{2} \cos{\left(\theta_{2}{\left(t \right)} \right)}}{2}\right) + 0.0416666666666667 l_{2}^{2} m_{2} \left(\frac{d}{d t} \theta_{2}{\left(t \right)}\right)^{2} + 0.5 m_{1} \left(\frac{d}{d t} d_{1}{\left(t \right)}\right)^{2} + 0.5 m_{2} \left(\frac{l_{2}^{2} \sin^{2}{\left(\theta_{2}{\left(t \right)} \right)} \left(\frac{d}{d t} \theta_{2}{\left(t \right)}\right)^{2}}{4} + \left(\frac{l_{2} \cos{\left(\theta_{2}{\left(t \right)} \right)} \frac{d}{d t} \theta_{2}{\left(t \right)}}{2} + \frac{d}{d t} d_{1}{\left(t \right)}\right)^{2}\right)$$

#### 5)
Herefter kan vi bruge Euler-Lagrange ligningen til at finde bevægelses ligninger:
$$
\frac{d}{dt}(\frac{\partial L}{\partial \dot{d_1}}) - \frac{\partial L}{\partial d_1} = F
$$
$$
\frac{d}{dt}(\frac{\partial L}{\partial \dot{\theta}}) - \frac{\partial L}{\partial \theta} = \tau

In [8]:
F = sp.diff(sp.diff(L, sp.diff(d1, t)), t) - sp.diff(L, d1)
tau = sp.diff(sp.diff(L, sp.diff(theta2, t)), t) - sp.diff(L, theta2)

F = F.subs({sp.diff(d1, t, t): sp.symbols('a'), sp.diff(d1, t): sp.symbols('v') ,sp.diff(theta2, t): sp.symbols('omega'), sp.diff(theta2, t, t): sp.symbols('alpha')})
tau = tau.subs({sp.diff(d1, t, t): sp.symbols('a'), sp.diff(d1, t): sp.symbols('v'), sp.diff(d1, t, t): sp.symbols('a'), sp.diff(theta2, t, t): sp.symbols('alpha')})  

lPrint("Bevægelses ligning for d1", F.factor().nsimplify())
lPrint("Bevægelses ligning for theta2", tau.factor().nsimplify())

**Bevægelses ligning for d1**: $$a m_{1} + a m_{2} + \frac{\alpha l_{2} m_{2} \cos{\left(\theta_{2}{\left(t \right)} \right)}}{2} - \frac{l_{2} m_{2} \omega^{2} \sin{\left(\theta_{2}{\left(t \right)} \right)}}{2}$$

**Bevægelses ligning for theta2**: $$\frac{l_{2} m_{2} \left(a \cos{\left(\theta_{2}{\left(t \right)} \right)} + \frac{\alpha l_{2} \sin^{2}{\left(\theta_{2}{\left(t \right)} \right)}}{2} + \frac{\alpha l_{2} \cos^{2}{\left(\theta_{2}{\left(t \right)} \right)}}{2} + \frac{\alpha l_{2}}{6} + g \sin{\left(\theta_{2}{\left(t \right)} \right)}\right)}{2}$$